# Problem W2D1: Baseline correction: transform, and why it is not agg

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dc00086-glitch/internship-prep/blob/main/problems/week02/w2d1_pandas_transform_baseline.ipynb)

| | |
|---|---|
| Problem ID | `W2D1` |
| Week / day | 2 / 1 |
| Topic | pandas: groupby.transform, baseline correction, alignment |
| Expected time | 20-30 min |

**How to work this notebook**

1. Run the setup cell to build the data.
2. Run the **Toolkit** cell. It demonstrates every move this problem needs, on a tiny made up example, so none of it is the answer. Read its output before you start typing.
3. Fill in every `# YOUR CODE` blank, one part at a time.
4. Run the check cell as often as you like. It tells you which part is wrong and why, without giving you the answer.
5. When you are done, run the last cell and copy the tutor block.

Stuck on something more basic, like what an array even is? The [foundations notebook](https://colab.research.google.com/github/dc00086-glitch/internship-prep/blob/main/reference/foundations.ipynb) covers Python, numpy, pandas and matplotlib from the ground up, and you can keep it open in a second tab.

Nothing here needs installing. Everything used is already in Colab.


## Scenario

A phase 1 study measured a target engagement biomarker in 15 subjects, five
each on placebo, a low dose and a high dose. Every subject was sampled at
baseline and at six timepoints after dosing.

The file arrives in assay run order, because that is how the bioanalytical lab
reports, so the rows are in no useful order at all.

Subjects start from very different baselines, which is normal for almost any
biomarker. Comparing raw values between groups would mostly tell you who
happened to start high. So the readout everyone actually reports is **percent
change from that subject's own baseline**, which means every row needs to know
its own subject's baseline value.

That is a different shape of problem from anything in week 1. A `groupby` that
summarises collapses 105 rows down to 15. Here you need to go the other way:
compute one number per subject, then put it back on all seven of that
subject's rows.

## agg collapses, transform aligns

This is the pandas idea that unlocks most real analysis work, and it is worth
being precise about.

`groupby(...).agg(...)` **collapses**. You hand it 105 rows and 15 subjects,
and you get 15 rows back, one per subject. The result has a different shape
from the input, so you cannot simply attach it as a new column.

`groupby(...).transform(...)` **aligns**. You hand it the same 105 rows, it
computes per group exactly like `agg` does, and then it broadcasts each
group's answer back out over every row of that group. You get 105 values back,
in the original row order, so it drops straight into a new column:

```python
df["SUBJ_MEAN"] = df.groupby("SUBJECT")["VALUE"].transform("mean")
```

Every row now carries its own subject's mean. No merge, no index juggling, no
risk of rows being reordered under you.

**Where this matters.** Any "relative to its own group" quantity is a
transform: percent change from baseline, deviation from a subject's own mean,
normalising each plate to its own control wells, each animal to its own
pre-dose value. Doing these with `agg` plus `merge` works, but it is three
steps instead of one and every one of them is a chance to lose or duplicate
rows.

**The catch with `"first"`.** To get a baseline you want the value at time
zero, and the natural shorthand is `transform("first")`, which takes the first
row of each group **as the data currently sits**. It does not look at your
time column and it has no idea what baseline means. If the rows are not sorted
so that time zero comes first within each subject, you will silently pick up
the wrong value, with no error. So you sort first, deliberately, and you check.

This is a recurring pattern in pandas: a convenient function that depends on
row order, used on data whose order nobody guaranteed. Sort first, every time.

**A third option worth knowing.** `groupby(...).apply(...)` can do anything,
because you hand it a whole chunk of the frame. It is also much slower and its
return shape is hard to predict. Reach for `agg` when you want a summary,
`transform` when you want to stay aligned, and `apply` only when neither fits.

## Your task

Three parts, worked one at a time.

| Part | What you work out | Variables to fill in |
|---|---|---|
| a | the data sorted, and each subject's baseline on every row | `n_subjects`, `df`, `baseline_ok` |
| b | percent change from baseline, post-dose rows only | `df_post` |
| c | the group level summary of that change | `summary` |

Below this you will find the **Setup** cell, then a **Toolkit** cell that
demonstrates every move on a tiny made up example, then **Part a**, **Part b**
and **Part c**, each with its own explanation and a cell to write in.

## Setup

Run this cell first. It builds the data with a fixed random seed, so your numbers match the checker and match mine.

In [ ]:
import numpy as np
import pandas as pd

PROBLEM_ID = "W2D1"
TOPIC = "pandas: groupby.transform, baseline correction, alignment"

rng = np.random.default_rng(2101)

times = np.array([0.0, 1.0, 2.0, 4.0, 8.0, 12.0, 24.0])
groups = np.repeat(["Placebo", "Low", "High"], 5)
subjects = ["S%02d" % i for i in range(1, 16)]
_effect = {"Placebo": 0.0, "Low": 0.28, "High": 0.55}

_rows = []
for _subj, _grp in zip(subjects, groups):
    _base = 100.0 * rng.lognormal(0.0, 0.30)          # baselines differ a lot
    _shape = np.exp(-0.09 * times) - np.exp(-1.1 * times)
    _shape = _shape / _shape.max()
    _true = _base * (1.0 - _effect[_grp] * _shape)
    _rows.append(np.round(_true * rng.lognormal(0.0, 0.05, size=times.size), 2))

raw = pd.DataFrame({
    "SUBJECT": np.repeat(subjects, times.size),
    "GROUP": np.repeat(groups, times.size),
    "TIME_H": np.tile(times, len(subjects)),
    "BIOMARKER": np.concatenate(_rows),
})

# The bioanalytical lab reports in assay run order, so the rows arrive scrambled.
raw = raw.sample(frac=1.0, random_state=7).reset_index(drop=True)

print("raw:", raw.shape)
print(raw.head(8))
print("")
print("note the row order: it is assay run order, which means nothing useful")


## Toolkit for this problem

| What you want to do | How |
|---|---|
| summarise, one row per group | `df.groupby("ID")["V"].agg("mean")` |
| the same number, on every row | `df.groupby("ID")["V"].transform("mean")` |
| the first row of each group | `.transform("first")`, which depends on row order |
| sort, and renumber the index | `df.sort_values([...], ignore_index=True)` |
| add a column from two others | `df["PCT"] = 100 * (df["A"] - df["B"]) / df["B"]` |
| keep some rows | `df[df["TIME_H"] > 0]` |
| check something is true everywhere | `(df["A"] == df["B"]).all()` |

`ignore_index=True` on `sort_values` is the same as following it with
`.reset_index(drop=True)`, just shorter.

Run the cell below, then carry on to **Part a**, which is the next section down.

In [ ]:
import numpy as np
import pandas as pd

# A tiny made up table. Nothing here is the answer.
demo = pd.DataFrame({
    "ID":   ["a", "b", "a", "b", "a", "b"],
    "TIME": [0.0, 0.0, 1.0, 1.0, 2.0, 2.0],
    "V":    [10.0, 50.0, 7.0, 40.0, 5.0, 25.0],
})
print("demo, sorted by TIME like the real data:")
print(demo)

print("")
print("--- 1. agg collapses, transform aligns ---")
print("agg gives one row per group:")
print(demo.groupby("ID")["V"].agg("mean"))
print("")
print("transform gives one value per ORIGINAL row:")
print(demo.groupby("ID")["V"].transform("mean"))
print("")
print("so transform drops straight into a column:")
demo["ID_MEAN"] = demo.groupby("ID")["V"].transform("mean")
print(demo)

print("")
print("--- 2. transform('first') depends on row order ---")
demo_sorted = demo.sort_values(["ID", "TIME"], ignore_index=True)
print("sorted by ID then TIME, so time 0 really is first in each group:")
print(demo_sorted)
demo_sorted["BASE"] = demo_sorted.groupby("ID")["V"].transform("first")
print("")
print(demo_sorted)
print("")
print("check it picked the time 0 value on every row:")
print((demo_sorted.loc[demo_sorted["TIME"] == 0, "V"].to_numpy()
       == demo_sorted.loc[demo_sorted["TIME"] == 0, "BASE"].to_numpy()).all())

print("")
print("--- 3. what happens if you forget to sort ---")
demo_bad = demo.sort_values(["TIME"], ascending=False, ignore_index=True)
demo_bad["BASE"] = demo_bad.groupby("ID")["V"].transform("first")
print("sorted with the LAST timepoint first, so 'first' picks the wrong value:")
print(demo_bad[["ID", "TIME", "V", "BASE"]])
print("no error, no warning, just wrong numbers all the way down")

print("")
print("--- 4. percent change, and keeping post dose rows ---")
demo_sorted["PCT"] = 100.0 * (demo_sorted["V"] - demo_sorted["BASE"]) / demo_sorted["BASE"]
print(demo_sorted[demo_sorted["TIME"] > 0][["ID", "TIME", "V", "BASE", "PCT"]])


## Part a. Sort, then put each subject's baseline on every row

The data arrived sorted by timepoint, which is exactly the order that makes
`transform("first")` give the right answer by luck rather than by design. Sort
it properly first, by subject and then by time, so that time zero really is
the first row of every subject.

Then add a `BASELINE` column holding that subject's time zero value, repeated
on all seven of their rows. This is a `transform`, not an `agg`: you want 105
values back, not 15.

Finally, prove it worked rather than assuming. Pull out the rows where
`TIME_H` is 0 and check that `BIOMARKER` and `BASELINE` agree on every one of
them. If the sort were wrong this check is what would catch it.

Fill in `n_subjects`, how many distinct subjects there are. `df`, the sorted
frame with its index renumbered and a `BASELINE` column added. And
`baseline_ok`, a single True or False.

Moves: `.nunique()`, `.sort_values([...], ignore_index=True)`,
`.groupby(...)["COL"].transform("first")`, and `(a == b).all()`.

In [ ]:
# Part a. Sort, then put each subject's baseline on every row.

n_subjects = None    # YOUR CODE: how many distinct subjects
df = None            # YOUR CODE: raw sorted by SUBJECT then TIME_H, index renumbered,
                     #            with a BASELINE column added
baseline_ok = None   # YOUR CODE: True if BIOMARKER equals BASELINE on every TIME_H == 0 row

print("n_subjects :", n_subjects)
print("baseline_ok:", baseline_ok)
if df is not None:
    print(df.head(8))


## Part b. Percent change from baseline

Now that every row knows its own subject's baseline, the change is one line of
arithmetic between two columns.

Percent change from baseline is `100 * (value - baseline) / baseline`. A
subject whose biomarker fell by a fifth gives `-20.0`. Note the sign: a drug
that suppresses the marker produces negative numbers, and reporting the
magnitude without the sign is a good way to confuse a reviewer.

Then drop the baseline rows themselves. At time zero the percent change is
exactly 0 for everybody by construction, so leaving those rows in a summary
adds a timepoint with no variability and drags the group means toward zero.

Fill in `df_post`: the post-dose rows only, so `TIME_H` greater than 0, with a
`PCT_CHANGE` column, index renumbered.

Moves: arithmetic between columns, boolean filtering,
`.reset_index(drop=True)`.

In [ ]:
# Part b. Percent change from baseline, post-dose rows only.

df_post = None   # YOUR CODE: TIME_H > 0 rows, with a PCT_CHANGE column, index renumbered

if df_post is not None:
    print(df_post.shape)
    print(df_post.head(6))


## Part c. Summarise the change by group and timepoint

Now collapse, because the question has changed. You are no longer asking about
individual rows, you are asking how each dose group behaved over time, so this
is an `agg`.

One row per `GROUP` and `TIME_H`, giving the mean percent change, its sample
SD across subjects, and how many subjects contributed. Keep an eye on that
count: it should be 5 in every cell, and if it is not, something upstream
dropped rows.

SD, never SEM. Pandas `std` already uses `n - 1`.

Fill in `summary`, a flat DataFrame with exactly these columns:

| column | meaning |
|---|---|
| `GROUP` | `High`, `Low` or `Placebo` |
| `TIME_H` | hours after dose |
| `PCT_MEAN` | mean percent change from baseline |
| `PCT_SD` | sample SD of percent change across subjects |
| `N` | how many subjects contributed |

Moves: `.groupby([...]).agg(NAME=("COL", "mean"), ...)`, `.reset_index()`.

In [ ]:
# Part c. Mean and SD percent change per group per timepoint.
# Columns wanted: GROUP, TIME_H, PCT_MEAN, PCT_SD, N

summary = None   # YOUR CODE

print(summary)


## Check your work

Run this cell. It prints PASS, FAIL or TODO for each part. Do not edit it.

In [ ]:
# Grading helpers. You do not need to read or edit this cell.
import base64 as _b64, json as _json, datetime as _dt, textwrap as _tw
import numpy as _np
import pandas as _pd

# Expected answers, encoded so that you do not read them by accident.
_EXP = _json.loads(_b64.b64decode("eyJkZl9iYXNlbGluZSI6IFs2OS4wMywgNjkuMDMsIDY5LjAzLCA2OS4wMywgNjkuMDMsIDY5LjAzLCA2OS4wMywgMTMzLjI4LCAxMzMuMjgsIDEzMy4yOCwgMTMzLjI4LCAxMzMuMjgsIDEzMy4yOCwgMTMzLjI4LCAxMTYuOTgsIDExNi45OCwgMTE2Ljk4LCAxMTYuOTgsIDExNi45OCwgMTE2Ljk4LCAxMTYuOTgsIDE5MS45OCwgMTkxLjk4LCAxOTEuOTgsIDE5MS45OCwgMTkxLjk4LCAxOTEuOTgsIDE5MS45OCwgMTY0LjA0LCAxNjQuMDQsIDE2NC4wNCwgMTY0LjA0LCAxNjQuMDQsIDE2NC4wNCwgMTY0LjA0LCAxNDcuNywgMTQ3LjcsIDE0Ny43LCAxNDcuNywgMTQ3LjcsIDE0Ny43LCAxNDcuNywgODUuMTcsIDg1LjE3LCA4NS4xNywgODUuMTcsIDg1LjE3LCA4NS4xNywgODUuMTcsIDk3LjAxLCA5Ny4wMSwgOTcuMDEsIDk3LjAxLCA5Ny4wMSwgOTcuMDEsIDk3LjAxLCA3MS40NSwgNzEuNDUsIDcxLjQ1LCA3MS40NSwgNzEuNDUsIDcxLjQ1LCA3MS40NSwgNzguMDMsIDc4LjAzLCA3OC4wMywgNzguMDMsIDc4LjAzLCA3OC4wMywgNzguMDMsIDUyLjkxLCA1Mi45MSwgNTIuOTEsIDUyLjkxLCA1Mi45MSwgNTIuOTEsIDUyLjkxLCAxNDguMDcsIDE0OC4wNywgMTQ4LjA3LCAxNDguMDcsIDE0OC4wNywgMTQ4LjA3LCAxNDguMDcsIDkwLjAyLCA5MC4wMiwgOTAuMDIsIDkwLjAyLCA5MC4wMiwgOTAuMDIsIDkwLjAyLCAyNTUuNzgsIDI1NS43OCwgMjU1Ljc4LCAyNTUuNzgsIDI1NS43OCwgMjU1Ljc4LCAyNTUuNzgsIDgwLjE5LCA4MC4xOSwgODAuMTksIDgwLjE5LCA4MC4xOSwgODAuMTksIDgwLjE5XSwgImRmX2Jhc2VsaW5lX3Vuc29ydGVkIjogWzY1LjU4LCA2NS41OCwgNjUuNTgsIDY1LjU4LCA2NS41OCwgNjUuNTgsIDY1LjU4LCAxMjQuNjMsIDEyNC42MywgMTI0LjYzLCAxMjQuNjMsIDEyNC42MywgMTI0LjYzLCAxMjQuNjMsIDEwOS4yOCwgMTA5LjI4LCAxMDkuMjgsIDEwOS4yOCwgMTA5LjI4LCAxMDkuMjgsIDEwOS4yOCwgMTczLjM0LCAxNzMuMzQsIDE3My4zNCwgMTczLjM0LCAxNzMuMzQsIDE3My4zNCwgMTczLjM0LCAxNDQuMTEsIDE0NC4xMSwgMTQ0LjExLCAxNDQuMTEsIDE0NC4xMSwgMTQ0LjExLCAxNDQuMTEsIDEwMy44MiwgMTAzLjgyLCAxMDMuODIsIDEwMy44MiwgMTAzLjgyLCAxMDMuODIsIDEwMy44MiwgNjYuNjUsIDY2LjY1LCA2Ni42NSwgNjYuNjUsIDY2LjY1LCA2Ni42NSwgNjYuNjUsIDk3LjAxLCA5Ny4wMSwgOTcuMDEsIDk3LjAxLCA5Ny4wMSwgOTcuMDEsIDk3LjAxLCA0Ny4zMSwgNDcuMzEsIDQ3LjMxLCA0Ny4zMSwgNDcuMzEsIDQ3LjMxLCA0Ny4zMSwgNTIuMDQsIDUyLjA0LCA1Mi4wNCwgNTIuMDQsIDUyLjA0LCA1Mi4wNCwgNTIuMDQsIDMwLjk3LCAzMC45NywgMzAuOTcsIDMwLjk3LCAzMC45NywgMzAuOTcsIDMwLjk3LCA5Ni42MSwgOTYuNjEsIDk2LjYxLCA5Ni42MSwgOTYuNjEsIDk2LjYxLCA5Ni42MSwgNTMuNzIsIDUzLjcyLCA1My43MiwgNTMuNzIsIDUzLjcyLCA1My43MiwgNTMuNzIsIDI1NS43OCwgMjU1Ljc4LCAyNTUuNzgsIDI1NS43OCwgMjU1Ljc4LCAyNTUuNzgsIDI1NS43OCwgNDUuMTQsIDQ1LjE0LCA0NS4xNCwgNDUuMTQsIDQ1LjE0LCA0NS4xNCwgNDUuMTRdLCAiZGZfbWVhbl9iYXNlbGluZSI6IFs2OC44Njg1NzE0Mjg1NzE0MywgNjguODY4NTcxNDI4NTcxNDMsIDY4Ljg2ODU3MTQyODU3MTQzLCA2OC44Njg1NzE0Mjg1NzE0MywgNjguODY4NTcxNDI4NTcxNDMsIDY4Ljg2ODU3MTQyODU3MTQzLCA2OC44Njg1NzE0Mjg1NzE0MywgMTI5LjAxODU3MTQyODU3MTQyLCAxMjkuMDE4NTcxNDI4NTcxNDIsIDEyOS4wMTg1NzE0Mjg1NzE0MiwgMTI5LjAxODU3MTQyODU3MTQyLCAxMjkuMDE4NTcxNDI4NTcxNDIsIDEyOS4wMTg1NzE0Mjg1NzE0MiwgMTI5LjAxODU3MTQyODU3MTQyLCAxMDguNDM4NTcxNDI4NTcxNDQsIDEwOC40Mzg1NzE0Mjg1NzE0NCwgMTA4LjQzODU3MTQyODU3MTQ0LCAxMDguNDM4NTcxNDI4NTcxNDQsIDEwOC40Mzg1NzE0Mjg1NzE0NCwgMTA4LjQzODU3MTQyODU3MTQ0LCAxMDguNDM4NTcxNDI4NTcxNDQsIDE3NS4wNzU3MTQyODU3MTQyNywgMTc1LjA3NTcxNDI4NTcxNDI3LCAxNzUuMDc1NzE0Mjg1NzE0MjcsIDE3NS4wNzU3MTQyODU3MTQyNywgMTc1LjA3NTcxNDI4NTcxNDI3LCAxNzUuMDc1NzE0Mjg1NzE0MjcsIDE3NS4wNzU3MTQyODU3MTQyNywgMTQ5LjM3NzE0Mjg1NzE0Mjg0LCAxNDkuMzc3MTQyODU3MTQyODQsIDE0OS4zNzcxNDI4NTcxNDI4NCwgMTQ5LjM3NzE0Mjg1NzE0Mjg0LCAxNDkuMzc3MTQyODU3MTQyODQsIDE0OS4zNzcxNDI4NTcxNDI4NCwgMTQ5LjM3NzE0Mjg1NzE0Mjg0LCAxMjguMTU3MTQyODU3MTQyODcsIDEyOC4xNTcxNDI4NTcxNDI4NywgMTI4LjE1NzE0Mjg1NzE0Mjg3LCAxMjguMTU3MTQyODU3MTQyODcsIDEyOC4xNTcxNDI4NTcxNDI4NywgMTI4LjE1NzE0Mjg1NzE0Mjg3LCAxMjguMTU3MTQyODU3MTQyODcsIDcyLjc3ODU3MTQyODU3MTQyLCA3Mi43Nzg1NzE0Mjg1NzE0MiwgNzIuNzc4NTcxNDI4NTcxNDIsIDcyLjc3ODU3MTQyODU3MTQyLCA3Mi43Nzg1NzE0Mjg1NzE0MiwgNzIuNzc4NTcxNDI4NTcxNDIsIDcyLjc3ODU3MTQyODU3MTQyLCA3OC45ODQyODU3MTQyODU3MiwgNzguOTg0Mjg1NzE0Mjg1NzIsIDc4Ljk4NDI4NTcxNDI4NTcyLCA3OC45ODQyODU3MTQyODU3MiwgNzguOTg0Mjg1NzE0Mjg1NzIsIDc4Ljk4NDI4NTcxNDI4NTcyLCA3OC45ODQyODU3MTQyODU3MiwgNjAuOTA0Mjg1NzE0Mjg1NzIsIDYwLjkwNDI4NTcxNDI4NTcyLCA2MC45MDQyODU3MTQyODU3MiwgNjAuOTA0Mjg1NzE0Mjg1NzIsIDYwLjkwNDI4NTcxNDI4NTcyLCA2MC45MDQyODU3MTQyODU3MiwgNjAuOTA0Mjg1NzE0Mjg1NzIsIDYyLjAwNzE0Mjg1NzE0Mjg2LCA2Mi4wMDcxNDI4NTcxNDI4NiwgNjIuMDA3MTQyODU3MTQyODYsIDYyLjAwNzE0Mjg1NzE0Mjg2LCA2Mi4wMDcxNDI4NTcxNDI4NiwgNjIuMDA3MTQyODU3MTQyODYsIDYyLjAwNzE0Mjg1NzE0Mjg2LCAzNS40MzU3MTQyODU3MTQyOCwgMzUuNDM1NzE0Mjg1NzE0MjgsIDM1LjQzNTcxNDI4NTcxNDI4LCAzNS40MzU3MTQyODU3MTQyOCwgMzUuNDM1NzE0Mjg1NzE0MjgsIDM1LjQzNTcxNDI4NTcxNDI4LCAzNS40MzU3MTQyODU3MTQyOCwgMTAyLjY0NTcxNDI4NTcxNDI4LCAxMDIuNjQ1NzE0Mjg1NzE0MjgsIDEwMi42NDU3MTQyODU3MTQyOCwgMTAyLjY0NTcxNDI4NTcxNDI4LCAxMDIuNjQ1NzE0Mjg1NzE0MjgsIDEwMi42NDU3MTQyODU3MTQyOCwgMTAyLjY0NTcxNDI4NTcxNDI4LCA2My4yMTI4NTcxNDI4NTcxNDYsIDYzLjIxMjg1NzE0Mjg1NzE0NiwgNjMuMjEyODU3MTQyODU3MTQ2LCA2My4yMTI4NTcxNDI4NTcxNDYsIDYzLjIxMjg1NzE0Mjg1NzE0NiwgNjMuMjEyODU3MTQyODU3MTQ2LCA2My4yMTI4NTcxNDI4NTcxNDYsIDE3Mi4wNjU3MTQyODU3MTQyOCwgMTcyLjA2NTcxNDI4NTcxNDI4LCAxNzIuMDY1NzE0Mjg1NzE0MjgsIDE3Mi4wNjU3MTQyODU3MTQyOCwgMTcyLjA2NTcxNDI4NTcxNDI4LCAxNzIuMDY1NzE0Mjg1NzE0MjgsIDE3Mi4wNjU3MTQyODU3MTQyOCwgNTMuNTUxNDI4NTcxNDI4NTcsIDUzLjU1MTQyODU3MTQyODU3LCA1My41NTE0Mjg1NzE0Mjg1NywgNTMuNTUxNDI4NTcxNDI4NTcsIDUzLjU1MTQyODU3MTQyODU3LCA1My41NTE0Mjg1NzE0Mjg1NywgNTMuNTUxNDI4NTcxNDI4NTddLCAiZGZfc3ViamVjdCI6IFsiUzAxIiwgIlMwMSIsICJTMDEiLCAiUzAxIiwgIlMwMSIsICJTMDEiLCAiUzAxIiwgIlMwMiIsICJTMDIiLCAiUzAyIiwgIlMwMiIsICJTMDIiLCAiUzAyIiwgIlMwMiIsICJTMDMiLCAiUzAzIiwgIlMwMyIsICJTMDMiLCAiUzAzIiwgIlMwMyIsICJTMDMiLCAiUzA0IiwgIlMwNCIsICJTMDQiLCAiUzA0IiwgIlMwNCIsICJTMDQiLCAiUzA0IiwgIlMwNSIsICJTMDUiLCAiUzA1IiwgIlMwNSIsICJTMDUiLCAiUzA1IiwgIlMwNSIsICJTMDYiLCAiUzA2IiwgIlMwNiIsICJTMDYiLCAiUzA2IiwgIlMwNiIsICJTMDYiLCAiUzA3IiwgIlMwNyIsICJTMDciLCAiUzA3IiwgIlMwNyIsICJTMDciLCAiUzA3IiwgIlMwOCIsICJTMDgiLCAiUzA4IiwgIlMwOCIsICJTMDgiLCAiUzA4IiwgIlMwOCIsICJTMDkiLCAiUzA5IiwgIlMwOSIsICJTMDkiLCAiUzA5IiwgIlMwOSIsICJTMDkiLCAiUzEwIiwgIlMxMCIsICJTMTAiLCAiUzEwIiwgIlMxMCIsICJTMTAiLCAiUzEwIiwgIlMxMSIsICJTMTEiLCAiUzExIiwgIlMxMSIsICJTMTEiLCAiUzExIiwgIlMxMSIsICJTMTIiLCAiUzEyIiwgIlMxMiIsICJTMTIiLCAiUzEyIiwgIlMxMiIsICJTMTIiLCAiUzEzIiwgIlMxMyIsICJTMTMiLCAiUzEzIiwgIlMxMyIsICJTMTMiLCAiUzEzIiwgIlMxNCIsICJTMTQiLCAiUzE0IiwgIlMxNCIsICJTMTQiLCAiUzE0IiwgIlMxNCIsICJTMTUiLCAiUzE1IiwgIlMxNSIsICJTMTUiLCAiUzE1IiwgIlMxNSIsICJTMTUiXSwgImRmX3RpbWUiOiBbMC4wLCAxLjAsIDIuMCwgNC4wLCA4LjAsIDEyLjAsIDI0LjAsIDAuMCwgMS4wLCAyLjAsIDQuMCwgOC4wLCAxMi4wLCAyNC4wLCAwLjAsIDEuMCwgMi4wLCA0LjAsIDguMCwgMTIuMCwgMjQuMCwgMC4wLCAxLjAsIDIuMCwgNC4wLCA4LjAsIDEyLjAsIDI0LjAsIDAuMCwgMS4wLCAyLjAsIDQuMCwgOC4wLCAxMi4wLCAyNC4wLCAwLjAsIDEuMCwgMi4wLCA0LjAsIDguMCwgMTIuMCwgMjQuMCwgMC4wLCAxLjAsIDIuMCwgNC4wLCA4LjAsIDEyLjAsIDI0LjAsIDAuMCwgMS4wLCAyLjAsIDQuMCwgOC4wLCAxMi4wLCAyNC4wLCAwLjAsIDEuMCwgMi4wLCA0LjAsIDguMCwgMTIuMCwgMjQuMCwgMC4wLCAxLjAsIDIuMCwgNC4wLCA4LjAsIDEyLjAsIDI0LjAsIDAuMCwgMS4wLCAyLjAsIDQuMCwgOC4wLCAxMi4wLCAyNC4wLCAwLjAsIDEuMCwgMi4wLCA0LjAsIDguMCwgMTIuMCwgMjQuMCwgMC4wLCAxLjAsIDIuMCwgNC4wLCA4LjAsIDEyLjAsIDI0LjAsIDAuMCwgMS4wLCAyLjAsIDQuMCwgOC4wLCAxMi4wLCAyNC4wLCAwLjAsIDEuMCwgMi4wLCA0LjAsIDguMCwgMTIuMCwgMjQuMF0sICJuX3Bvc3Rfcm93cyI6IDkwLCAibl9yYXdfcm93cyI6IDEwNSwgIm5fc3ViamVjdHMiOiAxNSwgInBvc3RfZmxpcHBlZCI6IFstNi4xNTY3NDM0NDQ4NzkwMzgsIDAuNDYzNTY2NTY1MjYxNDkxMiwgLTEuNjY1OTQyMzQzOTA4NDUzNywgMC4wNTc5NDU4MjA2NTc2OTQxMiwgNC45OTc4MjcwMzE3MjUzNDEsIDMuOTQwMzE1ODA0NzIyNTgzLCAtMC41NzAyMjgwOTEyMzY0ODc4LCAxLjEwMjk0MTE3NjQ3MDU4NzQsIDcuNzIwNTg4MjM1Mjk0MTIzLCA0LjMyOTIzMTY5MjY3NzA2OCwgMy4zMDg4MjM1Mjk0MTE3NjIzLCA2LjQ5MDA5NjAzODQxNTM3MSwgOC43ODc4MjY5Nzg5NzA3NjUsIDQuOTMyNDY3MDg4MzkxMTg3LCAxMS4xODEzOTg1Mjk2NjMxODgsIDE0LjExMzUyMzY3OTI2MTQxNiwgNS41MTM3NjMwMzY0MTY0ODMsIDYuNTgyMzIxNzY0NDA0MTczLCA4LjkzODQzMTA4NjU3MTUxNiwgMTIuODEzODM0Nzc0NDU1NjcxLCA5LjkyMjkwODYzNjMxNjI3NCwgOS42NjI0NjQ4NDAwODc1LCA5LjcwOTM0NDcyMzQwODY4MiwgMTAuNTg5NjQ0NzU0NjYxOTM2LCAxMi44MDE3NTU2NjkzNDg5NCwgMTIuMTQ5NDc1NzM3NjI0OTU2LCA3LjI2MDQyNDI4Njc1OTMyNCwgNi4wNDEyMDk0NjExMDcwNDUsIDkuMTMxOTE5MDQ0MTM1NTcyLCAxNS4xODUzMjA2NTM0OTkxNDUsIDI1LjE1OTEwNjI5NjU0NzA0NCwgMjkuNzA4ODY5MzI5NzIyNDEsIDE5LjEzMzM3ODQ2OTg3MTM1NiwgMTkuMzk3NDI3MjE3MzMyNDI0LCA0LjQ2MTc0Njc4NDAyMTY0OSwgLTUuMjQwMzUyMDY0OTk2NjIxLCAxOS42MzEzMjU1ODQxMjU4NjQsIDIwLjUwMDE3NjExODM1MTU0MywgMjYuMTI0MjIyMTQzOTQ3NCwgMjEuNzQ0NzQ1ODAyNTEyNjE1LCAxMS4zNjU1MDQyODU1NDY1NjIsIDIuNDc3Mzk4MTQ0ODg2Njk2NCwgMjkuMzE2NTY1MzAyNTQ2MTQsIDI5LjU3NDI3MDY5Mzc0MjkyNSwgMjguODMyMDc5MTY3MDk2MTc1LCAyMS4yMjQ2MTYwMTg5NjcxMiwgMTYuMjM1NDM5NjQ1Mzk3MzgsIDQuODg2MDk0MjE3MDkxMDMsIDIwLjQ4OTg1MzA0NDA4Njc3LCAyMS4zMjk2MDExMTk2NjQxMDUsIDMzLjc4NTg2NDI0MDcyNzc4LCAxNS4xMjk0NjExNjE2NTE1MDcsIDEwLjAzNDk4OTUwMzE0OTA1OCwgMi41NDcyMzU4MjkyNTEyMzUsIDI3LjgzNTQ0NzkwNDY1MjA1NywgMzMuMzA3NzAyMTY1ODMzNjUsIDI4LjM0ODA3MTI1NDY0NTY1MywgMjAuOTkxOTI2MTgyMjM3NjAyLCAxNy42OTgzMjExNTg1Mjg3NzQsIDE1LjU1ODExODY3MjMwNTUyNCwgNDEuNzY5MDQxNzY5MDQxNzcsIDU2Ljk2NDY1Njk2NDY1Njk2LCA1NC4wMzUxNTQwMzUxNTQwMywgNDEuNDY2NjQxNDY2NjQxNDcsIDIyLjMwMjAyMjMwMjAyMjI5OCwgMTQuNjQ3NTE0NjQ3NTE0NjUsIDQ2LjAxMjAyMTM0MTI1NzUxNSwgNTEuMDE2NDExMTU2ODg1MjU1LCA1MC43NDYyNjg2NTY3MTY0MSwgMzQuNzUzODMyNjQ2NzIxMTQsIDI2Ljg3MjQyNTIwNDI5NTI2LCA1LjM0MjA2Nzk0MDgzODc5LCAzOS4yMDIzOTk0NjY3ODUxNiwgNTEuNzU1MTY1NTE4NzczNjEsIDQ4LjY3ODA3MTUzOTY1Nzg0LCA0MC4zMjQzNzIzNjE2OTc0LCAyNC41Mzg5OTEzMzUyNTg4MiwgMy45NTQ2NzY3Mzg1MDI1NTgsIDQ1LjQ0NTMwNDU1ODYwNTA1LCA1NC40OTIxNDE2ODQyNTk5MSwgNTUuMjUwNjA1OTg5NTIyMjQsIDM0LjY3NDMyOTUwMTkxNTcxLCAzMS41MjMxODM5ODYyMzgxNzMsIDcuNzE3NTY5Nzg2NTM1MzA3LCA0NS4zMTczNzEyNDMyOTcxNywgNTUuOTU0NjA3ODA2NDU5NjYsIDU1Ljk1NDYwNzgwNjQ1OTY2LCA0My43MDg2OTE4NTY4NCwgMjguMDA4NDc5ODYwMzMxNzE1LCAzLjU5MTQ3MDI1ODEzNjkxOV0sICJwb3N0X2ZyYWMiOiBbMC4wNjE1Njc0MzQ0NDg3OTAzODQsIC0wLjAwNDYzNTY2NTY1MjYxNDkxMiwgMC4wMTY2NTk0MjM0MzkwODQ1NCwgLTAuMDAwNTc5NDU4MjA2NTc2OTQxMiwgLTAuMDQ5OTc4MjcwMzE3MjUzNDEsIC0wLjAzOTQwMzE1ODA0NzIyNTgzLCAwLjAwNTcwMjI4MDkxMjM2NDg3OCwgLTAuMDExMDI5NDExNzY0NzA1ODc0LCAtMC4wNzcyMDU4ODIzNTI5NDEyNCwgLTAuMDQzMjkyMzE2OTI2NzcwNjgsIC0wLjAzMzA4ODIzNTI5NDExNzYyLCAtMC4wNjQ5MDA5NjAzODQxNTM3LCAtMC4wODc4NzgyNjk3ODk3MDc2NCwgLTAuMDQ5MzI0NjcwODgzOTExODcsIC0wLjExMTgxMzk4NTI5NjYzMTg4LCAtMC4xNDExMzUyMzY3OTI2MTQxNywgLTAuMDU1MTM3NjMwMzY0MTY0ODMsIC0wLjA2NTgyMzIxNzY0NDA0MTczLCAtMC4wODkzODQzMTA4NjU3MTUxNiwgLTAuMTI4MTM4MzQ3NzQ0NTU2NywgLTAuMDk5MjI5MDg2MzYzMTYyNzQsIC0wLjA5NjYyNDY0ODQwMDg3NSwgLTAuMDk3MDkzNDQ3MjM0MDg2ODIsIC0wLjEwNTg5NjQ0NzU0NjYxOTM3LCAtMC4xMjgwMTc1NTY2OTM0ODk0LCAtMC4xMjE0OTQ3NTczNzYyNDk1NSwgLTAuMDcyNjA0MjQyODY3NTkzMjUsIC0wLjA2MDQxMjA5NDYxMTA3MDQ1LCAtMC4wOTEzMTkxOTA0NDEzNTU3MSwgLTAuMTUxODUzMjA2NTM0OTkxNDYsIC0wLjI1MTU5MTA2Mjk2NTQ3MDQ2LCAtMC4yOTcwODg2OTMyOTcyMjQxLCAtMC4xOTEzMzM3ODQ2OTg3MTM1NywgLTAuMTkzOTc0MjcyMTczMzI0MjUsIC0wLjA0NDYxNzQ2Nzg0MDIxNjQ5NSwgMC4wNTI0MDM1MjA2NDk5NjYyMSwgLTAuMTk2MzEzMjU1ODQxMjU4NjQsIC0wLjIwNTAwMTc2MTE4MzUxNTQzLCAtMC4yNjEyNDIyMjE0Mzk0NzQsIC0wLjIxNzQ0NzQ1ODAyNTEyNjEzLCAtMC4xMTM2NTUwNDI4NTU0NjU2MywgLTAuMDI0NzczOTgxNDQ4ODY2OTY1LCAtMC4yOTMxNjU2NTMwMjU0NjE0NCwgLTAuMjk1NzQyNzA2OTM3NDI5MjcsIC0wLjI4ODMyMDc5MTY3MDk2MTc1LCAtMC4yMTIyNDYxNjAxODk2NzExOCwgLTAuMTYyMzU0Mzk2NDUzOTczODEsIC0wLjA0ODg2MDk0MjE3MDkxMDMwNCwgLTAuMjA0ODk4NTMwNDQwODY3NzMsIC0wLjIxMzI5NjAxMTE5NjY0MTA0LCAtMC4zMzc4NTg2NDI0MDcyNzc4LCAtMC4xNTEyOTQ2MTE2MTY1MTUwOCwgLTAuMTAwMzQ5ODk1MDMxNDkwNTgsIC0wLjAyNTQ3MjM1ODI5MjUxMjM0OCwgLTAuMjc4MzU0NDc5MDQ2NTIwNTUsIC0wLjMzMzA3NzAyMTY1ODMzNjUsIC0wLjI4MzQ4MDcxMjU0NjQ1NjUsIC0wLjIwOTkxOTI2MTgyMjM3NjAyLCAtMC4xNzY5ODMyMTE1ODUyODc3NCwgLTAuMTU1NTgxMTg2NzIzMDU1MjQsIC0wLjQxNzY5MDQxNzY5MDQxNzcsIC0wLjU2OTY0NjU2OTY0NjU2OTYsIC0wLjU0MDM1MTU0MDM1MTU0MDMsIC0wLjQxNDY2NjQxNDY2NjQxNDcsIC0wLjIyMzAyMDIyMzAyMDIyMjk4LCAtMC4xNDY0NzUxNDY0NzUxNDY0OCwgLTAuNDYwMTIwMjEzNDEyNTc1MTQsIC0wLjUxMDE2NDExMTU2ODg1MjYsIC0wLjUwNzQ2MjY4NjU2NzE2NDEsIC0wLjM0NzUzODMyNjQ2NzIxMTQ0LCAtMC4yNjg3MjQyNTIwNDI5NTI2NCwgLTAuMDUzNDIwNjc5NDA4Mzg3OSwgLTAuMzkyMDIzOTk0NjY3ODUxNiwgLTAuNTE3NTUxNjU1MTg3NzM2LCAtMC40ODY3ODA3MTUzOTY1Nzg0LCAtMC40MDMyNDM3MjM2MTY5NzM5NywgLTAuMjQ1Mzg5OTEzMzUyNTg4MiwgLTAuMDM5NTQ2NzY3Mzg1MDI1NTgsIC0wLjQ1NDQ1MzA0NTU4NjA1MDU0LCAtMC41NDQ5MjE0MTY4NDI1OTkxLCAtMC41NTI1MDYwNTk4OTUyMjI0LCAtMC4zNDY3NDMyOTUwMTkxNTcwNywgLTAuMzE1MjMxODM5ODYyMzgxNzQsIC0wLjA3NzE3NTY5Nzg2NTM1MzA3LCAtMC40NTMxNzM3MTI0MzI5NzE2NiwgLTAuNTU5NTQ2MDc4MDY0NTk2NywgLTAuNTU5NTQ2MDc4MDY0NTk2NywgLTAuNDM3MDg2OTE4NTY4NCwgLTAuMjgwMDg0Nzk4NjAzMzE3MTQsIC0wLjAzNTkxNDcwMjU4MTM2OTE5XSwgInBvc3RfcGN0IjogWzYuMTU2NzQzNDQ0ODc5MDM4LCAtMC40NjM1NjY1NjUyNjE0OTEyLCAxLjY2NTk0MjM0MzkwODQ1MzcsIC0wLjA1Nzk0NTgyMDY1NzY5NDEyLCAtNC45OTc4MjcwMzE3MjUzNDEsIC0zLjk0MDMxNTgwNDcyMjU4MywgMC41NzAyMjgwOTEyMzY0ODc4LCAtMS4xMDI5NDExNzY0NzA1ODc0LCAtNy43MjA1ODgyMzUyOTQxMjMsIC00LjMyOTIzMTY5MjY3NzA2OCwgLTMuMzA4ODIzNTI5NDExNzYyMywgLTYuNDkwMDk2MDM4NDE1MzcxLCAtOC43ODc4MjY5Nzg5NzA3NjUsIC00LjkzMjQ2NzA4ODM5MTE4NywgLTExLjE4MTM5ODUyOTY2MzE4OCwgLTE0LjExMzUyMzY3OTI2MTQxNiwgLTUuNTEzNzYzMDM2NDE2NDgzLCAtNi41ODIzMjE3NjQ0MDQxNzMsIC04LjkzODQzMTA4NjU3MTUxNiwgLTEyLjgxMzgzNDc3NDQ1NTY3MSwgLTkuOTIyOTA4NjM2MzE2Mjc0LCAtOS42NjI0NjQ4NDAwODc1LCAtOS43MDkzNDQ3MjM0MDg2ODIsIC0xMC41ODk2NDQ3NTQ2NjE5MzYsIC0xMi44MDE3NTU2NjkzNDg5NCwgLTEyLjE0OTQ3NTczNzYyNDk1NiwgLTcuMjYwNDI0Mjg2NzU5MzI0LCAtNi4wNDEyMDk0NjExMDcwNDUsIC05LjEzMTkxOTA0NDEzNTU3MiwgLTE1LjE4NTMyMDY1MzQ5OTE0NSwgLTI1LjE1OTEwNjI5NjU0NzA0NCwgLTI5LjcwODg2OTMyOTcyMjQxLCAtMTkuMTMzMzc4NDY5ODcxMzU2LCAtMTkuMzk3NDI3MjE3MzMyNDI0LCAtNC40NjE3NDY3ODQwMjE2NDksIDUuMjQwMzUyMDY0OTk2NjIxLCAtMTkuNjMxMzI1NTg0MTI1ODY0LCAtMjAuNTAwMTc2MTE4MzUxNTQzLCAtMjYuMTI0MjIyMTQzOTQ3NCwgLTIxLjc0NDc0NTgwMjUxMjYxNSwgLTExLjM2NTUwNDI4NTU0NjU2MiwgLTIuNDc3Mzk4MTQ0ODg2Njk2NCwgLTI5LjMxNjU2NTMwMjU0NjE0LCAtMjkuNTc0MjcwNjkzNzQyOTI1LCAtMjguODMyMDc5MTY3MDk2MTc1LCAtMjEuMjI0NjE2MDE4OTY3MTIsIC0xNi4yMzU0Mzk2NDUzOTczOCwgLTQuODg2MDk0MjE3MDkxMDMsIC0yMC40ODk4NTMwNDQwODY3NywgLTIxLjMyOTYwMTExOTY2NDEwNSwgLTMzLjc4NTg2NDI0MDcyNzc4LCAtMTUuMTI5NDYxMTYxNjUxNTA3LCAtMTAuMDM0OTg5NTAzMTQ5MDU4LCAtMi41NDcyMzU4MjkyNTEyMzUsIC0yNy44MzU0NDc5MDQ2NTIwNTcsIC0zMy4zMDc3MDIxNjU4MzM2NSwgLTI4LjM0ODA3MTI1NDY0NTY1MywgLTIwLjk5MTkyNjE4MjIzNzYwMiwgLTE3LjY5ODMyMTE1ODUyODc3NCwgLTE1LjU1ODExODY3MjMwNTUyNCwgLTQxLjc2OTA0MTc2OTA0MTc3LCAtNTYuOTY0NjU2OTY0NjU2OTYsIC01NC4wMzUxNTQwMzUxNTQwMywgLTQxLjQ2NjY0MTQ2NjY0MTQ3LCAtMjIuMzAyMDIyMzAyMDIyMjk4LCAtMTQuNjQ3NTE0NjQ3NTE0NjUsIC00Ni4wMTIwMjEzNDEyNTc1MTUsIC01MS4wMTY0MTExNTY4ODUyNTUsIC01MC43NDYyNjg2NTY3MTY0MSwgLTM0Ljc1MzgzMjY0NjcyMTE0LCAtMjYuODcyNDI1MjA0Mjk1MjYsIC01LjM0MjA2Nzk0MDgzODc5LCAtMzkuMjAyMzk5NDY2Nzg1MTYsIC01MS43NTUxNjU1MTg3NzM2MSwgLTQ4LjY3ODA3MTUzOTY1Nzg0LCAtNDAuMzI0MzcyMzYxNjk3NCwgLTI0LjUzODk5MTMzNTI1ODgyLCAtMy45NTQ2NzY3Mzg1MDI1NTgsIC00NS40NDUzMDQ1NTg2MDUwNSwgLTU0LjQ5MjE0MTY4NDI1OTkxLCAtNTUuMjUwNjA1OTg5NTIyMjQsIC0zNC42NzQzMjk1MDE5MTU3MSwgLTMxLjUyMzE4Mzk4NjIzODE3MywgLTcuNzE3NTY5Nzg2NTM1MzA3LCAtNDUuMzE3MzcxMjQzMjk3MTcsIC01NS45NTQ2MDc4MDY0NTk2NiwgLTU1Ljk1NDYwNzgwNjQ1OTY2LCAtNDMuNzA4NjkxODU2ODQsIC0yOC4wMDg0Nzk4NjAzMzE3MTUsIC0zLjU5MTQ3MDI1ODEzNjkxOV0sICJzdW1tYXJ5X2NvbHMiOiBbIkdST1VQIiwgIlRJTUVfSCIsICJQQ1RfTUVBTiIsICJQQ1RfU0QiLCAiTiJdLCAic3VtbWFyeV9jb3VudCI6IFs1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1LCA1XSwgInN1bW1hcnlfZ3JvdXAiOiBbIkhpZ2giLCAiSGlnaCIsICJIaWdoIiwgIkhpZ2giLCAiSGlnaCIsICJIaWdoIiwgIkxvdyIsICJMb3ciLCAiTG93IiwgIkxvdyIsICJMb3ciLCAiTG93IiwgIlBsYWNlYm8iLCAiUGxhY2VibyIsICJQbGFjZWJvIiwgIlBsYWNlYm8iLCAiUGxhY2VibyIsICJQbGFjZWJvIl0sICJzdW1tYXJ5X21lYW4iOiBbLTQzLjU0OTIyNzY3NTc5NzMzLCAtNTQuMDM2NTk2NjI2MjA3MDcsIC01Mi45MzI5NDE2MDU1MDIwNCwgLTM4Ljk4NTU3MzU2Njc2MzE0NCwgLTI2LjY0OTAyMDUzNzYyOTI1LCAtNy4wNTA2NTk4NzQzMDU2NDQsIC0yNC40ODY0NTk2MjYzOTE1NzUsIC0yNi44ODQxMjM4ODU0NjI5MywgLTI3LjI0NDcyMzA1NTI1NzY3MywgLTE5LjY5NzYzNTI3NjU0MDI1NCwgLTExLjk1OTIwMDI3NTMyODY4NSwgLTQuMDQ1Njk4OTU5NzA3NTc0LCAtNC43NjAyMDg0Mzk3NTUxMzksIC02LjI5MjQ1NzA2ODQ0MDc3OCwgLTYuODgzODc1NDY4ODI0ODkxNSwgLTYuODQwODc1MDk4NzU4MTQ0LCAtNi41MzIzMzU0NzMwMTk1NjcsIC04LjU1NzUzOTgwMzE0MDY0Ml0sICJzdW1tYXJ5X24iOiAxOCwgInN1bW1hcnlfc2QiOiBbMi45NTE3MjcwMDI5ODk3MjM2LCAyLjU4Nzc3Mjg2OTY3Nzc3MjYsIDMuMTA2NTQwMDc0MTUwNTI1LCA0LjA4NTA0NTM4Mjg2MzY1NiwgMy41MDAxNDczMzY2ODk5ODM2LCA0LjU0NDk3OTcwNDI3NjgzMSwgNC4zMTY5MjI3MzU0NTk0NjcsIDUuNjU4NzUzMTA1NzY0NTEzNSwgNS4zMjg1MzI1ODMxMzgxNjIsIDIuNjk5ODA4MzM4NzgxMzQzNCwgNS4yNzk0NTU1MjQ0OTQ4NDMsIDcuNDg2MjgzNzAxODgwNDEyLCA3Ljg0MTIyMzI3MDE5MDk4MSwgNS45MDc0MDMxMzc0MzIzMDUsIDUuMDQwOTc0MzM0MTAzMDQ0LCA1LjMzNDA1ODgwNTYxMjYyLCAyLjc2NzQyMDE3NjYyMDYwMSwgNC40MDMxNjYzODkyODE1Nl0sICJzdW1tYXJ5X3NlbSI6IFsxLjMyMDA1MjQ0NTk0MTM0OTQsIDEuMTU3Mjg3MjA5Mzg1ODQwOCwgMS4zODkyODY5NTYxMjU1NjIzLCAxLjgyNjg4NzgzMzQ1MDk1OCwgMS41NjUzMTM0NzUyMjA3Mjk0LCAyLjAzMjU3NjcxNTAyMzk3NywgMS45MzA1ODY1MzgwMjAzNDE3LCAyLjUzMDY3MTMyMjQ3NTUwMTcsIDIuMzgyOTkyMjE1MjQzODk2LCAxLjIwNzM5MDk5NDM0NzE3MywgMi4zNjEwNDQyODczOTE0NTQ4LCAzLjM0Nzk2Nzg1MTI1MDY3NCwgMy41MDY3MDE2NTE3ODAwNDY0LCAyLjY0MTg3MDk5NzE1ODgzMzIsIDIuMjU0MzkyMjU2Nzc3MjI4NSwgMi4zODU0NjM2MTcwNjYyMzA3LCAxLjIzNzYyNzkyNzQ0NTYyNzQsIDEuOTY5MTU1ODcyNTM1MTczN10sICJzdW1tYXJ5X3RpbWUiOiBbMS4wLCAyLjAsIDQuMCwgOC4wLCAxMi4wLCAyNC4wLCAxLjAsIDIuMCwgNC4wLCA4LjAsIDEyLjAsIDI0LjAsIDEuMCwgMi4wLCA0LjAsIDguMCwgMTIuMCwgMjQuMF0sICJ1bnNvcnRlZF9kaWZmZXJzIjogdHJ1ZX0=").decode("utf-8"))

CHECK_RESULTS = []


class _Todo(Exception):
    """Raised when a part has not been attempted yet."""


def _need(name):
    g = globals()
    if name not in g:
        raise _Todo("`" + name + "` is not defined yet. Run the starter cell for this part.")
    val = g[name]
    if val is None:
        raise _Todo("`" + name + "` is still None. Fill in the # YOUR CODE blank.")
    return val


def _as_scalar(val):
    try:
        arr = _np.asarray(val, dtype=float)
    except Exception:
        return None
    if arr.size != 1:
        return None
    return float(arr.reshape(-1)[0])


def _as_array(val):
    try:
        return _np.asarray(val, dtype=float)
    except Exception:
        return None


def _num_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_scalar(a)
    if a is None:
        return False
    return bool(_np.isclose(a, float(b), rtol=rtol, atol=atol, equal_nan=True))


def _arr_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_array(a)
    if a is None:
        return False
    b = _np.asarray(b, dtype=float)
    if a.shape != b.shape:
        return False
    return bool(_np.allclose(a, b, rtol=rtol, atol=atol, equal_nan=True))


def _cols_of(obj):
    try:
        return [str(c) for c in obj.columns]
    except Exception:
        return None


def check():
    """Test every part and print PASS, FAIL or TODO for each one."""
    global CHECK_RESULTS
    names = sorted(n for n in globals() if n.startswith("_part_"))
    CHECK_RESULTS = []
    for name in names:
        label = name[len("_part_"):]
        try:
            problem = globals()[name]()
            status = "FAIL" if problem else "PASS"
            msg = problem or ""
        except _Todo as exc:
            status, msg = "TODO", str(exc)
        except Exception as exc:
            status = "FAIL"
            msg = "could not check this part, it raised %s: %s" % (type(exc).__name__, exc)
        CHECK_RESULTS.append((label, status, msg))
    _print_results()


def _print_results():
    bar = "=" * 70
    print(bar)
    print("Problem %s  |  %s" % (PROBLEM_ID, TOPIC))
    print(bar)
    for label, status, msg in CHECK_RESULTS:
        print("Part %s: %s" % (label, status))
        if msg:
            for line in _tw.wrap(msg, width=64):
                print("    " + line)
    n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
    print("-" * 70)
    print("Result: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    if n_pass == len(CHECK_RESULTS):
        print("All parts pass. Run the last cell to build your tutor block.")
    print(bar)


def send_to_tutor():
    """Build the block you copy to your tutor, along with your code."""
    lines = []
    lines.append("===== SEND TO TUTOR =====")
    lines.append("Problem ID : %s" % PROBLEM_ID)
    lines.append("Topic      : %s" % TOPIC)
    lines.append("Date       : %s" % _dt.date.today().isoformat())
    lines.append("Check results:")
    if not CHECK_RESULTS:
        lines.append("  check() has not been run yet. Run the check cell above first.")
    else:
        for label, status, msg in CHECK_RESULTS:
            lines.append("  Part %s: %s" % (label, status))
            if msg:
                for line in _tw.wrap(msg, width=58):
                    lines.append("      " + line)
        n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
        lines.append("Summary: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    lines.append("=========================")
    lines.append("Paste your code cells below this block.")
    return "\n".join(lines)



def _part_a():
    n_subjects = _need("n_subjects")
    df = _need("df")
    baseline_ok = _need("baseline_ok")

    got = _as_scalar(n_subjects)
    if got is None:
        return "`n_subjects` should be a single whole number."
    if int(round(got)) == _EXP["n_raw_rows"]:
        return ("`n_subjects` is the number of rows, not the number of subjects. Each "
                "subject appears once per timepoint.")
    if int(round(got)) != _EXP["n_subjects"]:
        return "`n_subjects` is not the number of distinct subjects in raw."

    if not isinstance(df, _pd.DataFrame):
        return "`df` should be a pandas DataFrame, but it is a %s." % type(df).__name__
    if "BASELINE" not in (_cols_of(df) or []):
        return "`df` has no BASELINE column yet."
    if len(df) != _EXP["n_raw_rows"]:
        return ("`df` has %d rows but it should still have all %d rows of raw. Part a "
                "adds a column, it does not remove rows." % (len(df), _EXP["n_raw_rows"]))
    if list(df.index) != list(range(len(df))):
        return ("`df` has not had its index renumbered. Pass ignore_index=True to "
                "sort_values, or follow with .reset_index(drop=True).")
    if [str(v) for v in df["SUBJECT"]] != _EXP["df_subject"]:
        return ("`df` is not sorted by SUBJECT then TIME_H. Sort before you transform, "
                "because transform('first') reads whatever row happens to be first.")
    if not _arr_close(df["TIME_H"].to_numpy(), _EXP["df_time"]):
        return "`df` is not in SUBJECT then TIME_H order."

    base = df["BASELINE"].to_numpy()
    if _arr_close(base, _EXP["df_mean_baseline"]):
        return ("`BASELINE` holds each subject's MEAN, not their time zero value. The "
                "aggregation you want is 'first', after sorting.")
    if _EXP["unsorted_differs"] and _arr_close(base, _EXP["df_baseline_unsorted"]):
        return ("`BASELINE` was taken with transform('first') on the UNSORTED data, so it "
                "picked whichever row happened to come first. Sort by SUBJECT and TIME_H "
                "before transforming.")
    if len(set(_np.round(base, 9))) == len(base):
        return ("`BASELINE` has a different value on every row, so it was not broadcast "
                "per subject. transform repeats one value across a whole group.")
    if not _arr_close(base, _EXP["df_baseline"]):
        return "`BASELINE` does not hold each subject's own time zero value on every row."

    if not isinstance(baseline_ok, (bool, _np.bool_)):
        val = _as_scalar(baseline_ok)
        if val is None:
            return ("`baseline_ok` should be a single True or False. Comparing two columns "
                    "gives a Series, so finish with .all().")
        baseline_ok = bool(val)
    if not bool(baseline_ok):
        return ("`baseline_ok` came out False. Either the baseline is wrong, or the check "
                "itself is comparing the wrong things.")
    return None


def _part_b():
    df_post = _need("df_post")
    if not isinstance(df_post, _pd.DataFrame):
        return "`df_post` should be a pandas DataFrame, but it is a %s." % type(df_post).__name__
    cols = _cols_of(df_post) or []
    if "PCT_CHANGE" not in cols:
        return "`df_post` has no PCT_CHANGE column yet."
    if "BASELINE" not in cols:
        return "`df_post` lost the BASELINE column. Build it from your df, not from raw."

    if len(df_post) == _EXP["n_raw_rows"]:
        return ("`df_post` still has the baseline rows. Drop TIME_H equal to 0, where the "
                "percent change is 0 for everybody by construction.")
    if len(df_post) != _EXP["n_post_rows"]:
        return ("`df_post` has %d rows but there should be %d post-dose rows."
                % (len(df_post), _EXP["n_post_rows"]))
    if (df_post["TIME_H"] <= 0).any():
        return "`df_post` still contains a TIME_H of 0 or less."
    if list(df_post.index) != list(range(len(df_post))):
        return "`df_post` has not had its index renumbered. Use .reset_index(drop=True)."

    got = df_post.sort_values(["SUBJECT", "TIME_H"])["PCT_CHANGE"].to_numpy()
    if _arr_close(got, _EXP["post_frac"]):
        return "`PCT_CHANGE` is a fraction. The task asks for a percent, so multiply by 100."
    if _arr_close(got, _EXP["post_flipped"]):
        return ("`PCT_CHANGE` has the sign the wrong way round. It is "
                "(value - baseline) / baseline, so a drug that suppresses the marker gives "
                "negative numbers.")
    if not _arr_close(got, _EXP["post_pct"]):
        return ("`PCT_CHANGE` is not 100 * (BIOMARKER - BASELINE) / BASELINE. Check that "
                "you divided by the baseline rather than by the current value.")
    return None


def _part_c():
    summary = _need("summary")
    if not isinstance(summary, _pd.DataFrame):
        return "`summary` should be a pandas DataFrame, but it is a %s." % type(summary).__name__
    cols = _cols_of(summary) or []
    for name in ("GROUP", "TIME_H"):
        if name not in cols and name in [str(n) for n in (summary.index.names or [])]:
            return "`%s` is in the index rather than a column. Call .reset_index()." % name
    missing = [c for c in _EXP["summary_cols"] if c not in cols]
    if missing:
        return "`summary` is missing the column(s) %s." % ", ".join(missing)

    if 0.0 in set(_np.asarray(summary["TIME_H"], dtype=float).tolist()):
        return ("`summary` includes the baseline timepoint, so it was built before the "
                "post-dose filter in part b.")
    if len(summary) != _EXP["summary_n"]:
        return ("`summary` has %d rows but there should be one per group and timepoint "
                "combination." % len(summary))

    got = summary.sort_values(["GROUP", "TIME_H"]).reset_index(drop=True)
    if [str(v) for v in got["GROUP"]] != _EXP["summary_group"]:
        return "`summary` does not cover the expected set of groups."
    if not _arr_close(got["TIME_H"].to_numpy(), _EXP["summary_time"]):
        return "`summary` does not cover the expected set of timepoints."
    if not _arr_close(got["N"].to_numpy(), _EXP["summary_count"]):
        return ("`N` is not the number of subjects in each cell. It should be 5 everywhere; "
                "the aggregation 'size' counts rows within the group.")
    if not _arr_close(got["PCT_MEAN"].to_numpy(), _EXP["summary_mean"]):
        return "`PCT_MEAN` is not the mean of PCT_CHANGE within each group and timepoint."
    if _arr_close(got["PCT_SD"].to_numpy(), _EXP["summary_sem"]):
        return ("`PCT_SD` is the SEM, not the SD. Something divided by the square root of "
                "n. This repo always reports SD.")
    if not _arr_close(got["PCT_SD"].to_numpy(), _EXP["summary_sd"]):
        return "`PCT_SD` is not the sample SD of PCT_CHANGE within each group and timepoint."
    return None


check()


## Send to tutor

In [ ]:
# Run this last. Copy everything it prints, then paste your own code under it.
print(send_to_tutor())
